In [46]:
import pandas as pd
import numpy as np
import re, unicodedata
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import os

In [47]:
ev2 = pd.read_csv('../../Datasets/evaluacion2.csv')
ev2.head(1)

,product_title,product_rating,is_best_seller,is_sponsored,buy_box_availability,sustainability_tags,has_coupon,discount_percentage,product_category,product_segment,log_original_price,log_purchased_last_month,log_total_reviews
0,"OWC 2.0TB Aura Pro X2 (GEN 4) SSD Complete Upgrade Solution Compatible with Mac Pro (Late 2013), High Performance NVMe Flash Upgrade, Including Tools, heatsink, and Envoy Pro Enclosure",4.4,No Badge,Sponsored,1,0,0,0.0,PC Components,Media,5.484755,0.0,5.883322


In [61]:
ev2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7717 entries, 0 to 7716
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   product_title             7717 non-null   object 
 1   product_rating            7717 non-null   float64
 2   is_best_seller            7717 non-null   object 
 3   is_sponsored              7717 non-null   object 
 4   buy_box_availability      7717 non-null   int64  
 5   sustainability_tags       7717 non-null   int64  
 6   has_coupon                7717 non-null   int64  
 7   discount_percentage       7717 non-null   float64
 8   product_category          7717 non-null   object 
 9   product_segment           7717 non-null   object 
 10  log_original_price        7717 non-null   float64
 11  log_purchased_last_month  7717 non-null   float64
 12  log_total_reviews         7717 non-null   float64
 13  brand_cat                 7717 non-null   object 
 14  is_high_

In [49]:
# 1. Definimos una lista de "Palabras que parecen marcas pero son ruido"
blacklist = ['EZ', 'THE', 'AND', 'FOR', 'PRO', 'NEW', 'OFF']

# 2. Tu proceso de extracción
ev2['brand'] = ev2['product_title'].str.split().str[0].str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)

# 3. FILTRO DE CALIDAD:
# Enviamos a OTHER si:
# - Está en la blacklist
# - Es un número puro (ej: "100")
# - Es demasiado corto (menos de 2 caracteres, excepto marcas como LG)
def clean_brand(b):
    if b in blacklist: return 'OTHER'
    if b.isdigit(): return 'OTHER' 
    if len(b) < 2: return 'OTHER'
    return b

ev2['brand'] = ev2['brand'].apply(clean_brand)

# 4. RE-CALCULAR EL TOP 50
# Tip: Si quieres que GIGABYTE sea 'OTHER' podrías sacarla del conteo manualmente
top_brands = ev2[ev2['brand'] != 'OTHER']['brand'].value_counts().nlargest(100).index

ev2['brand_cat'] = ev2['brand'].apply(lambda x: x if x in top_brands else 'OTHER')
ev2.drop(columns=['brand'], inplace=True)

In [50]:
# 2. BOOLEAN FEATURES (Regex)
# si es un modelo de alta gama
ev2['is_high_end'] = ev2['product_title'].str.contains(r'\b(Pro|Ultra|Gaming|Business|Elite|Max|Plus|Enterprise)\b', case=False, regex=True).astype(int)

C:\Users\diego\AppData\Local\Temp\ipykernel_32\984869603.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ev2['is_high_end'] = ev2['product_title'].str.contains(r'\b(Pro|Ultra|Gaming|Business|Elite|Max|Plus|Enterprise)\b', case=False, regex=True).astype(int)


In [51]:
def extraer_resolucion(title):
    title = str(title).lower()
    
    # 1. NIVEL TOP (8K, 5K - Muy caros)
    if re.search(r'\b(8k|5k)\b', title):
        return '8K_5K_Ultra'
        
    # 2. NIVEL ALTO (4K, UHD, Retina - Apple y Gama Alta)
    # "Retina" es clave para Apple, que no suele decir "4K" pero es caro.
    if re.search(r'\b(4k|uhd|2160p|ultra hd|retina)\b', title):
        return '4K_UHD_Retina'
        
    # 3. NIVEL MEDIO-ALTO (2K, QHD, 1440p - Gaming y Oficina Pro)
    if re.search(r'\b(2k|qhd|wqhd|1440p)\b', title):
        return '2K_QHD'
        
    # 4. NIVEL ESTÁNDAR (Full HD, 1080p - El estándar hoy en día)
    if re.search(r'\b(1080p|fhd|full hd|1920x1080)\b', title):
        return 'FHD_1080p'
        
    # 5. NIVEL BÁSICO (HD, 720p - Laptops baratos y TVs pequeñas)
    if re.search(r'\b(720p|hd|1366x768)\b', title):
        return 'HD_Basic'
        
    # Si no especifica nada (NaN para que XGBoost decida)
    return np.nan

In [52]:
def extractor_maestro(df):
    # Creamos copias para no tocar el original
    df_new = df.copy()
    
    # 1. Definimos las nuevas columnas vacías (NaN)
    specs = ['spec_ram_gb', 'spec_storage_gb', 'spec_screen_inch', 
    'spec_power_w', 'spec_pack_count', 'spec_resolution_cat',
    'spec_refresh_rate_hz', 'spec_ram_speed_mhz']
    for col in specs:
        df_new[col] = np.nan

    # 2. Lógica de Extracción Fila a Fila
    for i, row in df_new.iterrows():
        title = str(row['product_title']).lower()
        cat = str(row['product_category'])
        
        # --- BLOQUE A: RAM (Solo para Ordenadores/Tablets) ---
        if cat in ['Laptops', 'PC Components', 'Tablets & E-readers (DEVICES ONLY)', 'Computer Peripherals']:
            
            # EXPLICACIÓN REGEX RAM:
            # 1. (\d+)\s*(?:gb|g) -> Coge numero y unidad
            # 2. Grupo de Ruido con LOOKAHEAD NEGATIVO (?!) -> Acepta palabras intermedias SOLO SI:
            #    a) NO son 'ssd', 'hdd', 'storage', 'rom' (Evita coger discos)
            #    b) NO son 'd+ gb' (Evita coger otra capacidad que aparezca después)
            
            regex_ram = r'(\d+)\s*(?:gb|g)\s*(?:(?!(?:ssd|hdd|storage|rom)|\d+\s*(?:gb|g))[\w\-\.\(\)]+\s*){0,3}?(?:ram|memory|ddr|unified|vram|gddr)|(?:ram|memory|ddr\d?|capacity|vram|gddr\d?)\s*(?:kit)?\s*(\d+)\s*(?:gb|g)'
            
            match = re.search(regex_ram, title)
            if match:
                val = next((m for m in match.groups() if m is not None), None)
                if val: df_new.at[i, 'spec_ram_gb'] = float(val)
                
        # --- BLOQUE B: ALMACENAMIENTO (Ordenadores, Discos, Móviles) ---
        target_cats_storage = ['Laptops', 'PC Components', 'Storage & Memory Cards', 'Mobile Cell Phones & Smartphones', 'Tablets & E-readers (DEVICES ONLY)']
        if cat in target_cats_storage:
            
            # CAMBIO CLAVE EN EL REGEX:
            # Antes: (?!gb|tb|to) -> Solo miraba si empezaba por letras de unidad.
            # Ahora: (?!\d+\s*(?:gb|tb|to)) -> Mira si es un NÚMERO seguido de unidad.
            # Esto impide que "512GB" sea tratado como ruido.
            
            regex_storage = r'(\d+)\s*(gb|tb|to)\s*(?:(?!\d+\s*(?:gb|tb|to))[\w\-\.]+\s*){0,3}?(?:ssd|hdd|storage|flash|rom|emmc|hard drive|disk)'
            
            match = re.search(regex_storage, title)
            
            # Fallback para tarjetas de memoria
            if not match and cat == 'Storage & Memory Cards':
                match = re.search(r'(\d+)\s*(gb|tb|to)', title)
            
            if match:
                val = float(match.group(1))
                unit = match.group(2)
                
                if unit in ['tb', 'to']:
                    val *= 1024
                
                df_new.at[i, 'spec_storage_gb'] = val

        # C. PANTALLA Y RESOLUCIÓN (Visuales)
        if cat in ['Laptops', 'TV & Video Displays', 'Monitors', 'Tablets & E-readers (DEVICES ONLY)', 'Mobile Cell Phones & Smartphones', 'Office Supplies, Ink & Toner']:
            # Pulgadas
            match = re.search(r'(\d+(?:\.\d+)?)\s*(?:\"|inch|”|\-inch)', title)
            if match: df_new.at[i, 'spec_screen_inch'] = float(match.group(1))
            
            # Resolución (Nueva Lógica)
            res = extraer_resolucion(title)
            if pd.notna(res): df_new.at[i, 'spec_resolution_cat'] = res

        # --- BLOQUE D: POTENCIA (Cargadores, Audio, Componentes) ---
        if cat in ['Chargers, Adapters & Cables', 'Power & Batteries', 'Audio, Sound & Recording Gear', 'PC Components', 'Computer Peripherals']:
            # \b significa "Word Boundary" (Límite de palabra). 
            # Esto evita que 'Wireless', 'White', 'Warranty' o 'With' cuenten como 'W'.
            
            # Busca: Numero + (W o Watt o Vatios) + FINAL DE PALABRA
            match = re.search(r'(\d+)\s*(?:w|watt|vatios)\b', title)
            
            if match:
                df_new.at[i, 'spec_power_w'] = float(match.group(1))

        # --- BLOQUE E: PACKS (Oficina, Cables, Pilas) ---
        if cat in ['Office Supplies, Ink & Toner', 'Small Gadget Accessories (Cases & Protectors)', 
                   'Chargers, Adapters & Cables', 'Power & Batteries', 'Cameras & Photography', 
                   'Computer Peripherals', 'Other Electronics']:
            
            # 1. Patrón Estándar: "Pack of 2", "Set of 5"
            match = re.search(r'(?:pack of|set of|count of)\s*(\d+)', title)
            
            # 2. Patrón Inverso: "2-Pack", "10 count", "50 sheets", "2 pieces", "4 cartridges"
            # AÑADIDO: 'cartridges' a la lista de palabras clave
            if not match: 
                match = re.search(r'(\d+)\s*[-]?\s*(?:pack|set|count|pcs|sheets|pieces|cartridges)', title)
            
            # 3. Patrón Tinta Entre Paréntesis (CASO ESPECÍFICO PGBK)
            # Detecta: "(2 PGBK)", "(4 BK)", "(2 Black)", "(5 Cartridges)"
            # La clave son los paréntesis \(\) para no confundir con modelos (ej: Canon 280)
            if not match and cat == 'Office Supplies, Ink & Toner':
                match = re.search(r'\(\s*(\d+)\s*(?:pgbk|bk|c|m|y|black|cyan|magenta|yellow|color|ink)\b', title)

            if match:
                df_new.at[i, 'spec_pack_count'] = float(match.group(1))
            
            # 4. Caso especial Pair/Twin
            elif 'pair' in title or 'twin pack' in title: 
                df_new.at[i, 'spec_pack_count'] = 2.0
                
        # Buscamos Hz altos (90, 120, 144, 165, 240, 360, etc.)
        if cat in ['Laptops', 'TV & Video Displays', 'Monitors', 'Tablets & E-readers (DEVICES ONLY)', 'Mobile Cell Phones & Smartphones']:
            # Regex: Número + Hz (con boundary \b para no coger cosas raras)
            match_hz = re.search(r'(\d+)\s*hz\b', title)
            if match_hz:
                hz_val = float(match_hz.group(1))
                # FILTRO DE SEGURIDAD:
                # Ignoramos 50Hz o 60Hz si es una TV barata o Laptop, ya que es el estándar y a veces se confunde con el input eléctrico.
                # Nos interesan los valores "Gaming" que suben el precio.
                # O bien, lo guardamos todo y dejamos que XGBoost decida. Yo recomiendo guardarlo todo.
                df_new.at[i, 'spec_refresh_rate_hz'] = hz_val

        # 2. VELOCIDAD RAM (PC Components, Laptops)
        # Buscamos MHz (2666, 3200, 5200, 6000...)
        if cat in ['PC Components', 'Laptops', 'Computer Peripherals']:
            
            # Regex: Número + (mhz O mt/s O mts)
            # (?i) al principio hace que sea case-insensitive (aunque ya pasamos title.lower())
            match_speed = re.search(r'(\d+)\s*(?:mhz|mt/s|mts)\b', title)
            
            if match_speed:
                val = float(match_speed.group(1))
                
                # FILTRO ANTI-RUIDO:
                # A veces coge "2.4 ghz" (wifi) como 2 (mhz). 
                # Las RAMs modernas suelen ser de más de 1000 MHz.
                # Las viejas DDR2 eran 400-800. 
                # Vamos a poner un suelo de 200 para evitar coger frecuencias de radio o wifi mal escritas.
                if val > 200:
                    df_new.at[i, 'spec_ram_speed_mhz'] = val
        

    return df_new

In [53]:
ev2_specs = extractor_maestro(ev2)

C:\Users\diego\AppData\Local\Temp\ipykernel_32\4213627003.py:67: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'HD_Basic' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  if pd.notna(res): df_new.at[i, 'spec_resolution_cat'] = res


In [54]:
# 3. SEMANTIC FEATURE (TF-IDF + LSA 1 Component)
# Clean titles slightly for TF-IDF
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return text

ev2_specs['clean_title'] = ev2_specs['product_title'].apply(clean_text)

# CONFIGURACIÓN CLAVE: ngram_range=(1, 2)
# Esto captura palabras sueltas ("laptop") Y parejas ("gaming laptop")
tfidf = TfidfVectorizer(
    stop_words='english', 
    max_features=5000, 
    ngram_range=(1, 2),  # <--- AQUÍ ESTÁ LA MAGIA
    min_df=2             # Ignora erratas que aparecen solo 1 vez
)
tfidf_matrix = tfidf.fit_transform(ev2_specs['clean_title'])

# LSA de 15 componentes para digerir esos bigramas
n_components = 15
svd = TruncatedSVD(n_components=n_components, random_state=42)
lsa_matrix = svd.fit_transform(tfidf_matrix)

# ==============================================================================
# GUARDAR ARCHIVOS PKL (Aquí está lo nuevo)
# ==============================================================================

folder_path = 'Archivos_modelos'
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"📁 Carpeta '{folder_path}' creada.")
    
tfidf_path = os.path.join(folder_path, 'tfidf_vectorizer.pkl')
svd_path = os.path.join(folder_path, 'lsa_svd_model.pkl')

joblib.dump(tfidf, tfidf_path)
joblib.dump(svd, svd_path)
print("✅ Archivos 'tfidf_vectorizer.pkl' y 'lsa_svd_model.pkl' guardados.")
# ==============================================================================

# Añadimos al DF
lsa_cols = [f'lsa_{i}' for i in range(n_components)]
df_lsa = pd.DataFrame(lsa_matrix, columns=lsa_cols, index=ev2_specs.index)
df_final = pd.concat([ev2_specs, df_lsa], axis=1)

# 4. PREPARING FINAL DATAFRAME FOR XGBOOST
# Select original numerical columns and new engineered ones
cols_to_keep = [
    'product_title','product_rating', 'log_total_reviews', 'log_purchased_last_month', 'is_sponsored', 'buy_box_availability',
    'has_coupon', 'discount_percentage', 'product_category', 'brand_cat', 'is_high_end', 'spec_ram_gb', 'spec_storage_gb',
    'spec_screen_inch', 'spec_power_w', 'spec_pack_count', 'spec_resolution_cat', 'spec_refresh_rate_hz', 'spec_ram_speed_mhz', 'log_original_price', 'is_best_seller'
] + lsa_cols

df_engineered = df_final[cols_to_keep].copy()

# Convert strings to category type for XGBoost
df_engineered['product_category'] = df_engineered['product_category'].astype('category')
df_engineered['brand_cat'] = df_engineered['brand_cat'].astype('category')
df_engineered['spec_resolution_cat'] = df_engineered['spec_resolution_cat'].astype('category')
# Save to CSV
df_engineered.to_csv('../../Datasets/amazon_TF-IDF.csv', index=False)

print("Engineered DataFrame Head:")
print(df_engineered.head())
print("\nColumn Info:")
print(df_engineered.info())

📁 Carpeta 'Archivos_modelos' creada.
✅ Archivos 'tfidf_vectorizer.pkl' y 'lsa_svd_model.pkl' guardados.
Engineered DataFrame Head:
                                                                                                                                                                              product_title  \
0  OWC 2.0TB Aura Pro X2 (GEN 4) SSD Complete Upgrade Solution Compatible with Mac Pro (Late 2013), High Performance NVMe Flash Upgrade, Including Tools, heatsink, and Envoy Pro Enclosure   
1                                                                                                                              OWC 250GB Aura Pro 6G Flash SSD Upgrade for 2012 MacBook Air   
2                         HP 67XL Black High-yield Ink Cartridge | Works with HP DeskJet 1255, 2700, 4100 Series, HP ENVY 6000, 6400 Series | Eligible for Instant Ink | One Size | 3YM57AN   
3                          HP 67 Black/Tri-color Ink Cartridges for HP Printers | Works with Printer Seri

In [62]:
# --- DISPLAY ---
cols_to_show = ['product_title', 'brand_cat', 'is_high_end', 'spec_ram_gb', 'spec_storage_gb', 
                'spec_screen_inch', 'spec_power_w', 'spec_pack_count', 'spec_resolution_cat', 'spec_refresh_rate_hz', 'spec_ram_speed_mhz', 'log_original_price', 'product_category']

pd.set_option('display.max_colwidth', None)
df_engineered[cols_to_show].sample(10)

,product_title,brand_cat,is_high_end,spec_ram_gb,spec_storage_gb,spec_screen_inch,spec_power_w,spec_pack_count,spec_resolution_cat,spec_refresh_rate_hz,spec_ram_speed_mhz,log_original_price,product_category
6436,SANUS Extendable Soundbar Wall Mount for Sonos Arc & Sonos Arc Ultra Soundbar – 5” Depth Adjustment Optimized for Dolby Atmos - Black Speaker Mount,OTHER,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.709440,"Audio, Sound & Recording Gear"
4407,"Lexar 512GB NS100 SSD 2.5 Inch SATA III Internal Solid State Drive, Up to 550MB/s Read, Gray (LNS100-512RBNA)",OTHER,0,NaN,512.0,NaN,NaN,NaN,NaN,NaN,NaN,3.760968,Storage & Memory Cards
4387,"Legion Go USB-C Hub Dock for Legion Go – 6-in-1 Dock, Port Expansion, Charging, Video, Ethernet Support",OTHER,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.189503,Computer Peripherals
4210,"eufy Security, eufyCam 2C 3-Cam Kit, Security Camera Wireless Outdoor, Home Security System, HomeKit Compatibility, 1080p HD, IP67, Night Vision, Motion Only Alert, No Monthly Fee",OTHER,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.802088,Smart Home & Security
2535,"Philips 3-Outlet Extender, 2 Pack, Grounded Wall Tap, 3-Prong Adapter, Multiple Plug, Power Splitter, Cruise Essentials, Use for Home Office School Dorm, UL Listed, White, SPS1630W/37",PHILIPS,0,NaN,NaN,NaN,1630.0,2.0,NaN,NaN,NaN,2.250239,"Chargers, Adapters & Cables"
2695,"Woods E103 E-103 Wheel, Holds Up to 150 16/3 Extension 125 Feet of 14/3 Gauge Cord, Holiday, Rope, Hose Reel Storage and Light Wire, Heavy Duty Plastic, red and black",OTHER,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.183870,"Chargers, Adapters & Cables"
3086,Motorola Solutions T803 Waterproof IP54 Two Way Radio Walkie Talkie 35 mi. Bluetooth w/Charging Dock 2-Pack (Lime Green),MOTOROLA,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.824868,Mobile Cell Phones & Smartphones
2416,Racing Wheel Overdrive Designed for Xbox Series X|S By HORI - Officially Licensed by Microsoft,OTHER,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.791816,PC Components
1945,"ESR for AirPods 4 Case, Compatible with AirPods 4th Generation Case (2024), Compatible with MagSafe, Powerful Drop Protection, Magnetic Lid, Cyber Series, Black",ESR,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.638343,Small Gadget Accessories (Cases & Protectors)
6467,"ASUS ROG Strix G16 (2025) Gaming Laptop, 16” ROG Nebula Display 16:10 2.5K 240Hz/3ms, NVIDIA® GeForce RTX™ 5070 Laptop GPU, Intel® Core™ Ultra 9 275HX, 32GB DDR5, 2TB Gen 4 SSD, Wi-Fi 7, Win 11 Pro",ASUS,1,32.0,2048.0,16.0,NaN,NaN,8K_5K_Ultra,240.0,NaN,7.762171,Laptops


In [63]:
cols_check = ['spec_ram_gb', 'spec_storage_gb', 'spec_screen_inch', 
              'spec_power_w', 'spec_pack_count', 'spec_resolution_cat', 
              'spec_refresh_rate_hz', 'spec_ram_speed_mhz']

print("--- PORCENTAJE DE DATOS EXTRAÍDOS (NO NULOS) ---")
for col in cols_check:
    if col in df_engineered.columns:
        non_null = df_engineered[col].count()
        total = len(df_engineered)
        pct = (non_null / total) * 100
        print(f"{col}: {pct:.1f}% cubierto ({non_null} productos)")

--- PORCENTAJE DE DATOS EXTRAÍDOS (NO NULOS) ---
spec_ram_gb: 6.9% cubierto (532 productos)
spec_storage_gb: 8.5% cubierto (658 productos)
spec_screen_inch: 12.9% cubierto (994 productos)
spec_power_w: 6.1% cubierto (468 productos)
spec_pack_count: 7.8% cubierto (599 productos)
spec_resolution_cat: 8.3% cubierto (644 productos)
spec_refresh_rate_hz: 3.1% cubierto (236 productos)
spec_ram_speed_mhz: 1.3% cubierto (100 productos)


In [64]:

# 1. Asumiendo que tienes 'lsa_matrix' (tus vectores LSA) y 'y' (log_prices)
# Si no los tienes a mano, recupéralos de tu df_final
# lsa_cols = [c for c in df_final.columns if 'lsa_' in c]
# lsa_matrix = df_final[lsa_cols].values
# prices = np.expm1(df_final['log_original_price']).values # Precios reales en euros
# titles = df_final['product_title'].values

# (Simulación para que entiendas la lógica, ajusta con tus variables reales)
# Vamos a coger una muestra aleatoria de 1000 productos para no tardar una eternidad
n_samples = 1000
if len(df_final) > n_samples:
    indices = np.random.choice(len(df_final), n_samples, replace=False)
else:
    indices = np.arange(len(df_final))

subset_lsa = df_final.iloc[indices][[c for c in df_final.columns if 'lsa_' in c]].values
subset_prices = np.expm1(df_final.iloc[indices]['log_original_price']).values
subset_titles = df_final.iloc[indices]['product_title'].values

print("Calculando similitudes...")
# Matriz de similitud (1 = idénticos, 0 = nada que ver)
sim_matrix = cosine_similarity(subset_lsa)

print("Buscando 'Falsos Gemelos' (Texto igual, Precio distinto)...")
print("-" * 80)

falsos_gemelos = []

# Recorremos la matriz (solo la mitad superior para no repetir)
for i in range(len(indices)):
    for j in range(i + 1, len(indices)):
        similarity = sim_matrix[i, j]
        
        # SI SON MUY PARECIDOS EN TEXTO (> 0.95)
        if similarity > 0.95:
            price_a = subset_prices[i]
            price_b = subset_prices[j]
            
            # PERO EL PRECIO ES MUY DISTINTO (Uno vale el triple que el otro)
            ratio = max(price_a, price_b) / (min(price_a, price_b) + 1e-6)
            
            if ratio > 3.0: # Umbral de escándalo
                falsos_gemelos.append({
                    'Similitud': similarity,
                    'Producto A': subset_titles[i][:50] + "...",
                    'Precio A': f"{price_a:.2f} €",
                    'Producto B': subset_titles[j][:50] + "...",
                    'Precio B': f"{price_b:.2f} €",
                    'Ratio Precio': ratio
                })

# Convertir a DataFrame y ordenar por el ratio de precio más bestia
df_falsos = pd.DataFrame(falsos_gemelos).sort_values('Ratio Precio', ascending=False).head(20)

print(df_falsos.to_string())

Calculando similitudes...
Buscando 'Falsos Gemelos' (Texto igual, Precio distinto)...
--------------------------------------------------------------------------------
     Similitud                                             Producto A   Precio A                                             Producto B   Precio B  Ratio Precio
243   0.981772  Sony NEW Alpha 7S III Full-frame Interchangeable L...  3798.00 €  BAGSMART Digital Camera Case, Waterproof & Protect...    16.99 €    223.543248
353   0.953723  Canon EOS R5 C Mirrorless Camera (Body Only), 45 M...  3399.00 €  BAGSMART Digital Camera Case, Waterproof & Protect...    16.99 €    200.058846
7     0.976724  PGYTECH Camera Wrist Strap for Photographers Adjus...    29.95 €  Sony NEW Alpha 7S III Full-frame Interchangeable L...  3798.00 €    126.811348
9     0.988940  PGYTECH Camera Wrist Strap for Photographers Adjus...    29.95 €  Canon EOS R5 C Mirrorless Camera (Body Only), 45 M...  3399.00 €    113.489145
14    0.963306  PGYTECH Came